# Seven encodings in Verovio, and an LLM writes ABC

Session 1, *Why encode music?*, parts 8 and 10 (Demos B and D). The same demos run on the course page; here you can change the inputs and keep your work.

[Verovio](https://www.verovio.org) reads many text encodings of music and renders them as notation. We give it the first bars of *Alle meine Entchen* in six encodings, plus a chant fragment, and then check a melody written by a language model.

**How to use this notebook:** run the cells from top to bottom with Shift+Enter. Edit the text between the triple quotes `"""` and run the cell again. If something goes wrong, the message appears under the cell; the notebook keeps running.

## Setup

The first cell installs what the notebook needs; run it once after each server start. The second cell defines helper functions; it is collapsed because you do not need to read it.

In [ ]:
%pip install -q verovio==6.3.0 mido matplotlib ipywidgets bokeh

In [ ]:
# Helper functions for this notebook. Run this cell; you do not need to read it.
# They follow the same rules as the demos on the course page.

# Read pitches, metre and key from the MEI that Verovio writes.
#
# Why not Verovio's own MIDI values? Verovio 6.3 drops gestural accidentals
# when it imports **kern: with *k[f#c#], the token 4f# becomes
# <note pname="f"/> and getMIDIValuesForElement() reports F natural.
# So we apply the key signature ourselves:
#   accid.ges > accid > accidental earlier in the bar > key signature.

import re

STEPS = {"c": 0, "d": 2, "e": 4, "f": 5, "g": 7, "a": 9, "b": 11}
ACCID = {"s": 1, "f": -1, "ss": 2, "x": 2, "ff": -2, "n": 0, "xs": 3, "ts": 3, "tf": -3}
SHARP_ORDER = "fcgdaeb"
FLAT_ORDER = "beadgcf"


def _attrs(text):
    out = {}
    for m in re.finditer(r'([\w.:]+)="([^"]*)"', text, re.ASCII):
        out[m.group(1)] = m.group(2)
    return out


def _key_from_sig(sig):
    # "2s" -> {f: 1, c: 1}; "0" -> {}
    m = re.match(r"^(\d+)([sf]?)$", sig or "")
    key = {}
    if not m:
        return key
    n = int(m.group(1))
    order = FLAT_ORDER if m.group(2) == "f" else SHARP_ORDER
    for i in range(n):
        key[order[i]] = -1 if m.group(2) == "f" else 1
    return key


# Map xml:id -> MIDI pitch for every <note> with pname and oct.
def pitches_from_mei(mei):
    pitches = {}
    key = {}
    carried = {}  # "staff:pname:oct" -> alteration, reset per measure
    staff = "1"
    note = None  # open <note> waiting for a child <accid>

    def finish(n):
        where = f"{staff}:{n['pname']}:{n['oct']}"
        if n.get("ges") is not None:
            alter = n["ges"]
        elif n.get("written") is not None:
            alter = n["written"]
        elif where in carried:
            alter = carried[where]
        else:
            alter = key.get(n["pname"], 0)
        if n.get("written") is not None:
            carried[where] = n["written"]
        pitches[n["id"]] = 12 * (int(n["oct"]) + 1) + STEPS[n["pname"]] + alter

    for m in re.finditer(r"<(/?)([\w]+)\b([^>]*?)(/?)>", mei, re.ASCII):
        closing, tag, body, self_closing = m.group(1), m.group(2), m.group(3), m.group(4)
        if closing:
            if tag == "note" and note:
                finish(note)
                note = None
            continue
        a = _attrs(body)
        if "keysig" in a:
            key = _key_from_sig(a["keysig"])
        if tag == "keySig" and "sig" in a:
            key = _key_from_sig(a["sig"])
        if tag == "measure":
            carried = {}
        if tag == "staff" and a.get("n"):
            staff = a["n"]
        if tag == "note" and a.get("pname") and "oct" in a:
            n = {
                "id": a.get("xml:id"),
                "pname": a["pname"],
                "oct": a["oct"],
                "ges": ACCID.get(a.get("accid.ges")),
                "written": ACCID.get(a.get("accid")),
            }
            if self_closing:
                finish(n)
            else:
                note = n
        if tag == "accid" and note:
            if "accid.ges" in a:
                note["ges"] = ACCID.get(a["accid.ges"])
            if "accid" in a:
                note["written"] = ACCID.get(a["accid"])
    return pitches


# Metre {count, unit}; 4/4 if the MEI has none.
def meter_from_mei(mei):
    tag = re.search(r"<meterSig\b[^>]*>", mei, re.ASCII)
    source = tag.group(0) if tag else mei
    count = re.search(r'\b(?:meter\.)?count="(\d+)"', source, re.ASCII)
    unit = re.search(r'\b(?:meter\.)?unit="(\d+)"', source, re.ASCII)
    if count and unit:
        return {"count": int(count.group(1)), "unit": int(unit.group(1))}
    return {"count": 4, "unit": 4}


# First key signature as fifths: 2 = two sharps, -3 = three flats.
def fifths_from_mei(mei):
    m = re.search(r'\bkeysig="(\d+)([sf]?)"', mei, re.ASCII) or re.search(
        r'<keySig\b[^>]*\bsig="(\d+)([sf]?)"', mei, re.ASCII
    )
    if not m:
        return 0
    n = int(m.group(1))
    return -n if m.group(2) == "f" else n

# Melody <-> binary pitch x time grid (Demo C). Pure functions, no DOM.
# The Python version in 01_binary_grid.ipynb follows the same rules.
#
# A cell covers [i / res, (i + 1) / res) quarter notes. A note fills every
# cell whose start lies inside the note. Notes that cover no cell start are lost.

import math

NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]


def pitch_name(midi):
    return NAMES[midi % 12] + str(math.floor(midi / 12) - 1)


def is_black_key(midi):
    return len(NAMES[midi % 12]) > 1


# Notes from a Verovio timemap: [{pitch, on, off}] in quarter notes.
# pitches: dict xml:id -> MIDI pitch (pitches_from_mei in mei.py); ids not in
# the map (rests) are skipped.
def notes_from_timemap(timemap, pitches):
    offs = {}
    for event in timemap:
        for id_ in event.get("off") or []:
            offs[id_] = event["qstamp"]
    notes = []
    for event in timemap:
        for id_ in event.get("on") or []:
            if id_ not in pitches:
                continue
            notes.append({
                "pitch": pitches[id_],
                "on": event["qstamp"],
                "off": offs.get(id_, event["qstamp"]),
            })
    notes.sort(key=lambda n: (n["on"], n["pitch"]))
    return notes


def bar_quarters(meter):
    return (meter["count"] * 4) / meter["unit"]


# Start of the cell grid: round to avoid 0.9999 artefacts.
EPS = 1e-9


def _js_round(x):
    # Math.round is round-half-up; Python round() is banker's rounding.
    return math.floor(x + 0.5)


# to_grid(notes, res, meter) — res and meter are positional (JS passes {res, meter}).
def to_grid(notes, res, meter):
    bar_cells = _js_round(bar_quarters(meter) * res)
    end = max([0] + [n["off"] for n in notes])
    bars = max(1, math.ceil(end / bar_quarters(meter) - EPS))
    n_cols = bars * bar_cells
    pitch_list = [n["pitch"] for n in notes]
    # Math.min(...[]) is Infinity; Math.max(...[]) is -Infinity.
    low = min(pitch_list) if pitch_list else float("inf")
    high = max(pitch_list) if pitch_list else float("-inf")
    occ = {}
    ons = {}
    p = low
    while p <= high:
        occ[p] = [0] * n_cols
        ons[p] = [0] * n_cols
        p += 1
    lost = []
    for n in notes:
        first = math.ceil(n["on"] * res - EPS)
        stop = math.ceil(n["off"] * res - EPS)  # first cell not covered
        if first >= stop:
            lost.append(n)
            continue
        i = first
        while i < stop and i < n_cols:
            occ[n["pitch"]][i] = 1
            i += 1
        ons[n["pitch"]][first] = 1
    return {
        "res": res,
        "meter": meter,
        "barCells": bar_cells,
        "nCols": n_cols,
        "low": low,
        "high": high,
        "occ": occ,
        "ons": ons,
        "lost": lost,
    }


# Rows to show, highest first. compact: hide black-key rows that stay empty.
def grid_rows(grid, compact=True):
    rows = []
    p = grid["high"]
    while p >= grid["low"]:
        if compact and is_black_key(p) and not any(grid["occ"][p]):
            p -= 1
            continue
        rows.append(p)
        p -= 1
    return rows


# Grid as text, one row per pitch, cells grouped by bar (as on the slides).
def grid_to_text(grid, compact=True, onsets=False):
    rows = grid_rows(grid, compact)
    bars = grid["nCols"] / grid["barCells"]
    width = max(grid["barCells"], 5) + 2

    def label(s):
        return s.ljust(6)

    def block(cells):
        header = label("") + "".join(
            f"bar {b + 1}".ljust(width) for b in range(int(bars))
        ).rstrip()
        lines = [header]
        for p in rows:
            line = label(pitch_name(p))
            for b in range(int(bars)):
                chunk = cells[p][b * grid["barCells"]:(b + 1) * grid["barCells"]]
                line += "".join(str(v) for v in chunk).ljust(width)
            lines.append(line.rstrip())
        return "\n".join(lines)

    if not onsets:
        return block(grid["occ"])
    # Both grids in one: a note can only start where a pitch sounds, so three values are enough.
    three = {p: [(2 if grid["ons"][p][i] else 1) if v else 0 for i, v in enumerate(grid["occ"][p])] for p in rows}
    return (
        f"Sounding (1 = the pitch sounds):\n{block(grid['occ'])}\n\n"
        f"Onsets (1 = a note starts):\n{block(grid['ons'])}\n\n"
        f"Both in one grid (2 = a note starts, 1 = it continues, 0 = silence):\n{block(three)}"
    )


# Read the grid back as events: a new event starts whenever the set of
# sounding pitches changes or (with onsets) a sounding pitch has an onset.
def grid_to_events(grid, onsets=False):
    events = []
    current = None
    for i in range(grid["nCols"]):
        active = []
        attack = False
        p = grid["low"]
        while p <= grid["high"]:
            if grid["occ"][p][i]:
                active.append(p)
                if onsets and grid["ons"][p][i]:
                    attack = True
            p += 1
        key = ",".join(str(x) for x in active)
        if current and current["key"] == key and not attack:
            current["len"] += 1
        else:
            current = {"key": key, "start": i, "len": 1, "pitches": active}
            events.append(current)
    return [{"start": e["start"], "len": e["len"], "pitches": e["pitches"]} for e in events]


def events_to_notes(events, res):
    notes = []
    for e in events:
        for p in e["pitches"]:
            notes.append({
                "pitch": p,
                "on": e["start"] / res,
                "off": (e["start"] + e["len"]) / res,
            })
    notes.sort(key=lambda n: (n["on"], n["pitch"]))
    return notes


# kern pitch: c = C4, cc = C5, C = C3, CC = C2.
def kern_pitch(midi):
    name = NAMES[midi % 12]
    octave = math.floor(midi / 12) - 1
    letter = name[0]
    accidental = "#" if len(name) > 1 else ""
    repeat = octave - 3 if octave >= 4 else 4 - octave
    base = letter.lower() if octave >= 4 else letter
    return base * repeat + accidental


# Durations kern can write as one token, in quarter notes, largest first.
KERN_DURATIONS = [
    (6, "1."), (4, "1"), (3, "2."), (2, "2"), (1.5, "4."), (1, "4"),
    (0.75, "8."), (0.5, "8"), (0.375, "16."), (0.25, "16"), (0.125, "32"),
]


def _split_duration(quarters):
    parts = []
    left = quarters
    while left > EPS:
        fit = next(((q, recip) for q, recip in KERN_DURATIONS if q <= left + EPS), None)
        if not fit:
            break
        parts.append(fit)
        left -= fit[0]
    return parts


# Events (from grid_to_events) -> a **kern score Verovio can render.
def events_to_kern(events, grid):
    res, meter, bar_cells = grid["res"], grid["meter"], grid["barCells"]
    sounding = [p for e in events for p in e["pitches"]]
    sorted_p = sorted(sounding)
    median = sorted_p[len(sorted_p) // 2] if sorted_p else 60
    lines = [
        "**kern",
        "*clefF4" if median < 57 else "*clefG2",
        f"*M{meter['count']}/{meter['unit']}",
    ]
    # Cut each event at bar lines, then into writable durations.
    pieces = []
    for e in events:
        start = e["start"]
        end = e["start"] + e["len"]
        chunks = []
        while start < end:
            bar_end = (math.floor(start / bar_cells) + 1) * bar_cells
            stop = min(end, bar_end)
            for q, recip in _split_duration((stop - start) / res):
                chunks.append({"recip": recip, "start": start, "bar": math.floor(start / bar_cells)})
            start = stop
        for k, c in enumerate(chunks):
            if len(chunks) == 1:
                tie = ""
            elif k == 0:
                tie = "["
            elif k == len(chunks) - 1:
                tie = "]"
            else:
                tie = "_"
            pieces.append({**c, "pitches": e["pitches"], "tie": tie})
    bar = -1
    for piece in pieces:
        if piece["bar"] != bar:
            bar = piece["bar"]
            lines.append(f"={bar + 1}")
        if not piece["pitches"]:
            lines.append(f"{piece['recip']}r")
            continue
        tokens = []
        for p in piece["pitches"]:
            token = piece["recip"] + kern_pitch(p)
            if piece["tie"] == "[":
                tokens.append("[" + token)
            else:
                tokens.append(token + piece["tie"])
        lines.append(" ".join(tokens))
    lines.extend(["==", "*-"])
    return "\n".join(lines) + "\n"


# Compare two note lists: same pitches at the same times?
# Key by (pitch, on, off) so 1 and 1.0 match (JS string keys would differ).
def compare_notes(before, after):
    def key(n):
        return (n["pitch"], n["on"], n["off"])

    after_keys = {key(n) for n in after}
    before_keys = {key(n) for n in before}
    return {
        "kept": sum(1 for n in before if key(n) in after_keys),
        "missing": [n for n in before if key(n) not in after_keys],
        "added": [n for n in after if key(n) not in before_keys],
    }

# Input checks before Verovio sees the text, and melody checks for Demo D.
# Pure functions, no DOM. The notebooks carry the same rules in Python.

import re


# Some broken inputs crash Verovio 6.3 instead of giving an error, and some
# valid ones are misread (tested 2026-10-14). errors: do not load the text;
# warnings: load it, but say what Verovio will get wrong.
def precheck(fmt, text):
    problems = []
    warnings = []
    if fmt == "abc":
        lines = re.split(r"\r?\n", text)
        if not any(re.match(r"^X:", l) for l in lines):
            problems.append("ABC needs an X: line (for example X:1) before the tune.")
        if not any(re.match(r"^K:", l) for l in lines):
            problems.append("ABC needs a K: line (the key, for example K:C) before the notes.")
        depth = 0
        unmatched = False
        chord_length = False
        for line in lines:
            if re.match(r"^[A-Za-z]:", line):
                continue  # header and lyrics lines
            body = re.sub(r"%.*$", "", line)
            body = re.sub(r'"[^"]*"', "", body)
            body = re.sub(r"\|\]|\[\|", "|", body)
            if re.search(r"\][\d/]", body):
                chord_length = True
            for ch in body:
                if ch == "[":
                    depth += 1
                if ch == "]":
                    if depth == 0:
                        unmatched = True
                    else:
                        depth -= 1
        if unmatched:
            problems.append("There is a ] without a matching [ in the notes. Use |] for a final bar line.")
        if chord_length:
            warnings.append(
                "Verovio 6.3 ignores a length written after a chord, as in [CEG]2, and plays it as one unit. "
                "Write the length inside: [C2E2G2]."
            )
    if fmt == "humdrum":
        lines = [l for l in re.split(r"\r?\n", text) if l and not l.startswith("!!")]
        head = next((l for l in lines if l.startswith("**")), None)
        if head and re.search(r" {2,}|\*\*\w+ +\*\*", head, re.ASCII) and "\t" not in head:
            problems.append("Humdrum columns (spines) must be separated by tabs, not spaces.")
        manipulators = any(
            any(re.match(r"^\*[\^v+x]$", t) for t in l.split("\t"))
            for l in lines
        )
        if head and not manipulators:
            width = len(head.split("\t"))
            bad = next((i for i, l in enumerate(lines) if len(l.split("\t")) != width), -1)
            if bad >= 0:
                n_cols = len(lines[bad].split("\t"))
                problems.append(
                    f'Line "{lines[bad]}" has {n_cols} column(s), '
                    f"but the file has {width}. Every line needs one entry per spine."
                )
    return {"errors": problems, "warnings": warnings}


# LLM answers often come inside a Markdown code fence: keep only its content.
def clean_answer(text):
    fence = re.search(r"```[\w-]*\n([\s\S]*?)```", text, re.ASCII)
    return ((fence.group(1) if fence else text).strip() + "\n")


# Bar lengths in quarter notes, from the timemap's measure starts.
def bar_lengths(timemap, notes):
    starts = [e["qstamp"] for e in timemap if e.get("measureOn")]
    end = max(
        [0]
        + [e["qstamp"] for e in timemap]
        + [n["off"] for n in notes]
    )
    return [
        (starts[i + 1] if i + 1 < len(starts) else end) - s
        for i, s in enumerate(starts)
    ]


def parse_pitch(name):
    m = re.match(r"^([A-Ga-g])([#b]?)(-?\d)$", name.strip())
    if not m:
        return None
    step = {"c": 0, "d": 2, "e": 4, "f": 5, "g": 7, "a": 9, "b": 11}[m.group(1).lower()]
    alter = 1 if m.group(2) == "#" else -1 if m.group(2) == "b" else 0
    return 12 * (int(m.group(3)) + 1) + step + alter


def _js_num(n):
    # JS template strings write 2 for 2.0 and 3.5 for 3.5.
    if isinstance(n, float) and n.is_integer():
        return str(int(n))
    return str(n)


# Compare what Verovio understood with what the prompt asked for.
# found: {meter, fifths, bars: [lengths], notes}; spec: {meter, fifths, bars, low, high, final}
def melody_checks(found, spec):
    checks = []

    def add(label, ok, detail):
        checks.append({"label": label, "ok": ok, "detail": detail})

    want = (spec["meter"]["count"] * 4) / spec["meter"]["unit"]
    add(
        f"Metre is {spec['meter']['count']}/{spec['meter']['unit']}",
        found["meter"]["count"] == spec["meter"]["count"] and found["meter"]["unit"] == spec["meter"]["unit"],
        f"found {found['meter']['count']}/{found['meter']['unit']}",
    )
    add(f"{spec['bars']} bars", len(found["bars"]) == spec["bars"], f"found {len(found['bars'])}")
    wrong = [[i + 1, q] for i, q in enumerate(found["bars"]) if abs(q - want) > 1e-6]
    add(
        f"Every bar adds up to {_js_num(want)} quarter notes",
        len(wrong) == 0,
        ", ".join(f"bar {b}: {_js_num(q)}" for b, q in wrong) if wrong else "all bars complete",
    )

    def sig(f):
        if f == 0:
            return "no sharps or flats"
        if f > 0:
            return f"{f} sharp{'s' if f > 1 else ''}"
        return f"{-f} flat{'s' if f < -1 else ''}"

    add(f"Key signature: {sig(spec['fifths'])}", found["fifths"] == spec["fifths"], f"found {sig(found['fifths'])}")
    if found["notes"]:
        pitches = [n["pitch"] for n in found["notes"]]
        low = min(pitches)
        high = max(pitches)
        add(
            f"Range within {pitch_name(spec['low'])}–{pitch_name(spec['high'])}",
            low >= spec["low"] and high <= spec["high"],
            f"found {pitch_name(low)}–{pitch_name(high)}",
        )
        last = found["notes"][-1]["pitch"]
        final_name = re.sub(r"-?\d+$", "", pitch_name(spec["final"]))
        add(
            f"Ends on {final_name}",
            last % 12 == spec["final"] % 12,
            f"last note {pitch_name(last)}",
        )
    else:
        add("Has notes", False, "Verovio found no notes")
    return checks

# Verovio in a separate Python process.
#
# Some broken inputs crash Verovio 6.3 instead of giving an error (tested
# 2026-10-14): ABC without X: or K:, kern with too few columns. In a notebook
# a crash kills the kernel, and all your variables are gone. So Verovio runs
# in a child process: if it crashes, only the child dies. The child's error
# output is Verovio's log, which Jupyter would otherwise not show.

import json
import math
import os
import subprocess
import sys

from IPython.display import HTML, display

VEROVIO_VERSION = "6.3.0"

_CHILD = r'''
import json, sys
import verovio
job = json.loads(sys.stdin.read())
tk = verovio.toolkit()
tk.setOptions(job["options"])
out = {"ok": bool(tk.loadData(job["text"])), "version": tk.getVersion()}
if out["ok"]:
    out["svg"] = [tk.renderToSVG(p) for p in range(1, tk.getPageCount() + 1)] if job["svg"] else []
    out["mei"] = tk.getMEI()
    out["timemap"] = tk.renderToTimemap({"includeMeasures": True, "includeRests": True})
sys.stdout.write("\n@@RESULT@@\n" + json.dumps(out))
'''


# Load text in one format and return what the demos need (as on the web page).
# width: target width in pixels; Verovio lays the music out to fit it.
def read_score(fmt, text, width=800, scale=40, header="auto", svg=True):
    checked = precheck(fmt, text)
    result = {"ok": False, "errors": checked["errors"], "warnings": checked["warnings"],
              "log": "", "svg": [], "mei": "", "timemap": [], "notes": []}
    if result["errors"]:
        return result
    options = {
        "inputFrom": fmt, "scale": scale, "pageWidth": max(600, math.floor(width * 100 / scale + 0.5)),
        "adjustPageHeight": True, "pageMarginTop": 30, "pageMarginBottom": 30,
        "pageMarginLeft": 30, "pageMarginRight": 30, "header": header, "footer": "none",
    }
    child = subprocess.run(
        [sys.executable, "-c", _CHILD],
        input=json.dumps({"text": text, "options": options, "svg": svg}),
        capture_output=True, text=True, encoding="utf-8", timeout=300,
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    result["log"] = child.stderr.strip()
    if child.returncode < 0:
        result["errors"].append(
            f"Verovio crashed on this text (signal {-child.returncode}). This happens with some broken "
            "input that Verovio does not report as an error. Undo your last change and try again.")
        return result
    if child.returncode > 0 or "\n@@RESULT@@\n" not in child.stdout:
        result["errors"].append("The Verovio process failed; see the log below. Did the install cell run?")
        return result
    out = json.loads(child.stdout.rsplit("\n@@RESULT@@\n", 1)[1])
    if not out["ok"]:
        result["errors"].append(f"Verovio could not read this text as {fmt}.")
        return result
    result.update(ok=True, version=out["version"], svg=out["svg"], mei=out["mei"], timemap=out["timemap"])
    result["notes"] = notes_from_timemap(result["timemap"], pitches_from_mei(result["mei"]))
    result["meter"] = meter_from_mei(result["mei"])
    result["fifths"] = fifths_from_mei(result["mei"])
    return result


# The score on a white background (readable in dark themes), then the messages.
def show(result):
    for page in result["svg"]:
        display(HTML(f'<div style="background:white; padding:4px; overflow-x:auto">{page}</div>'))
    for text in result["errors"]:
        print("Error:", text)
    for text in result["warnings"]:
        print("Warning:", text)
    for line in result["log"].splitlines():
        if line.strip():
            print("Verovio:", line)


def render(fmt, text, **options):
    result = read_score(fmt, text, **options)
    show(result)
    return result

# Piano roll: time from left to right, pitch from bottom to top, one bar per note.
# The same view CAMAT uses later for its note tables.
#
# BACKEND chooses how rolls are drawn, as CAMAT's backend="plt" / "bokeh" does:
#   "plt"    a picture, drawn by matplotlib
#   "bokeh"  an interactive plot: drag to move, turn the mouse wheel to zoom,
#            point at a note to see its pitch, times and loudness.

import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch, Rectangle

BACKEND = "plt"
TRACK_COLORS = ["#1f77b4", "#d62728", "#9467bd", "#ff7f0e", "#2ca02c", "#8c564b"]
_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
_UNITS = {"s": "time (seconds)", "q": "time (quarter notes)"}
_bokeh_loaded = False  # run this cell again after reloading the browser tab


def _pitch_label(p):
    return f"{_NAMES[p % 12]}{p // 12 - 1}"


def _color(n):
    return TRACK_COLORS[n.get("track", 0) % len(TRACK_COLORS)]


# notes: [{"pitch", "start", "end"}], optionally "track" (colour), "role" (legend),
#        "alpha" (opacity) and "velocity" (shown when you point at a note).
# bands: [{"start", "end", "row", "color", "label"}] drawn under the keyboard (pedals).
# bar: draw a line every `bar` time units (for scores: quarter notes per bar).
# backend: "plt" or "bokeh"; None uses BACKEND.
def draw_roll(notes, start=None, end=None, low=None, high=None, bands=(), bar=None,
              unit="s", title=None, width=12, height=None, backend=None):
    if start is None:
        start = min((n["start"] for n in notes), default=0)
    if end is None:
        end = max((n["end"] for n in notes), default=start + 1)
    pitches = [n["pitch"] for n in notes]
    if low is None:
        low = min(pitches) - 1 if pitches else 59
    if high is None:
        high = max(pitches) + 1 if pitches else 72
    rows = sorted({b["row"] for b in bands})
    if height is None:
        height = min(8, max(2.5, 0.12 * (high - low + 2 * len(rows)) + 1))
    backend = (backend or BACKEND).strip().lower()
    if backend == "plt":
        _roll_plt(notes, start, end, low, high, bands, rows, bar, unit, title, width, height)
    elif backend == "bokeh":
        _roll_bokeh(notes, start, end, low, high, bands, rows, bar, unit, title, height)
    else:
        raise ValueError(f"Unknown backend {backend!r}: use 'plt' or 'bokeh'.")


def _roll_plt(notes, start, end, low, high, bands, rows, bar, unit, title, width, height):
    below = 2 * len(rows)
    fig, ax = plt.subplots(figsize=(width, height))
    for p in range(low, high + 1):
        if p % 12 in (1, 3, 6, 8, 10):
            ax.axhspan(p - 0.5, p + 0.5, color="0.93", lw=0, zorder=0)
    shortest = (end - start) / 1500
    rects = [Rectangle((n["start"], n["pitch"] - 0.4), max(n["end"] - n["start"], shortest), 0.8) for n in notes]
    colors = [to_rgba(_color(n), n.get("alpha", 0.9)) for n in notes]
    ax.add_collection(PatchCollection(rects, facecolors=colors, edgecolors="none", zorder=2))
    ticks = [(p, _pitch_label(p)) for p in range(low, high + 1) if p % 12 == 0]
    for b in bands:
        y = low - 1.5 - 2 * rows.index(b["row"])
        ax.add_patch(Rectangle((b["start"], y - 0.6), b["end"] - b["start"], 1.2, color=b["color"], lw=0))
    for row in rows:
        label = next(b["label"] for b in bands if b["row"] == row)
        ticks.append((low - 1.5 - 2 * rows.index(row), label))
    if bar:
        t = bar * (start // bar)
        while t <= end:
            ax.axvline(t, color="0.6", lw=0.8, zorder=1)
            t += bar
    roles = {}
    for n in notes:
        if n.get("role"):
            roles.setdefault(n["role"], _color(n))
    if roles:
        ax.legend(handles=[Patch(color=c, label=r) for r, c in roles.items()], loc="upper left", fontsize=8)
    ax.set_yticks([y for y, _ in ticks], [label for _, label in ticks])
    ax.set_xlim(start, end)
    ax.set_ylim(low - 0.5 - below, high + 0.5)
    ax.set_xlabel(_UNITS.get(unit, unit))
    ax.set_ylabel("pitch")
    ax.grid(axis="y", color="0.85", lw=0.6)
    if title:
        ax.set_title(title)
    fig.tight_layout()
    plt.show()


def _roll_bokeh(notes, start, end, low, high, bands, rows, bar, unit, title, height):
    global _bokeh_loaded
    from bokeh.io import output_notebook, show
    from bokeh.models import ColumnDataSource, FixedTicker, HoverTool, Range1d, Span
    from bokeh.plotting import figure
    from bokeh.resources import INLINE

    if not _bokeh_loaded:
        output_notebook(resources=INLINE, hide_banner=True)  # Bokeh's JavaScript, once per page
        _bokeh_loaded = True
    below = 2 * len(rows)
    fig = figure(
        title=title, height=int(90 * height), sizing_mode="stretch_width",
        x_range=Range1d(start, end, bounds=(start, end)),
        y_range=Range1d(low - 0.5 - below, high + 0.5, bounds=(low - 0.5 - below, high + 0.5)),
        x_axis_label=_UNITS.get(unit, unit), y_axis_label="pitch",
        tools="xpan,xwheel_zoom,box_zoom,reset,save", active_drag="xpan", active_scroll="xwheel_zoom",
    )
    fig.xgrid.grid_line_color = None
    keys = [p for p in range(low, high + 1) if p % 12 in (1, 3, 6, 8, 10)]
    fig.hbar(y=keys, left=start, right=end, height=1, color="black", alpha=0.06)
    if bar:
        t = bar * (start // bar)
        while t <= end:
            fig.add_layout(Span(location=t, dimension="height", line_color="#999999", line_width=1))
            t += bar

    shortest = (end - start) / 1500
    groups = {}
    for n in notes:
        groups.setdefault((n.get("track", 0), n.get("role")), []).append(n)
    for (track, role), group in sorted(groups.items(), key=lambda item: item[0][0]):
        data = {
            "left": [n["start"] for n in group],
            "right": [max(n["end"], n["start"] + shortest) for n in group],
            "bottom": [n["pitch"] - 0.4 for n in group],
            "top": [n["pitch"] + 0.4 for n in group],
            "alpha": [n.get("alpha", 0.9) for n in group],
            "name": [_pitch_label(n["pitch"]) for n in group],
            "start": [n["start"] for n in group],
            "end": [n["end"] for n in group],
        }
        tips = [("pitch", "@name"), ("start", "@start{0.000}"), ("end", "@end{0.000}")]
        if all("velocity" in n for n in group):
            data["velocity"] = [n["velocity"] for n in group]
            tips.append(("velocity", "@velocity"))
        legend = {"legend_label": role} if role else {}
        glyph = fig.quad(source=ColumnDataSource(data), left="left", right="right", bottom="bottom", top="top",
                         fill_color=_color(group[0]), fill_alpha="alpha", line_color=None, **legend)
        fig.add_tools(HoverTool(renderers=[glyph], tooltips=tips))
    if fig.legend:
        fig.legend.location = "top_left"
        fig.legend.click_policy = "hide"  # click a track in the legend to hide it

    ticks = {p: _pitch_label(p) for p in range(low, high + 1) if p % 12 == 0}
    for b in bands:
        y = low - 1.5 - 2 * rows.index(b["row"])
        fig.quad(left=b["start"], right=b["end"], bottom=y - 0.6, top=y + 0.6, color=b["color"], line_color=None)
        ticks[y] = b["label"]
    fig.yaxis.ticker = FixedTicker(ticks=sorted(ticks))
    fig.yaxis.major_label_overrides = ticks
    show(fig)


# Piano roll of a score read by read_score(): time in quarter notes, a line per bar.
def score_roll(result, **options):
    notes = [{"pitch": n["pitch"], "start": n["on"], "end": n["off"]} for n in result["notes"]]
    meter = result.get("meter") or {"count": 4, "unit": 4}
    draw_roll(notes, start=0, bar=meter["count"] * 4 / meter["unit"], unit="q", **options)

# Demo D: render an LLM's answer and check it against what the prompt asked for.
# spec: {"meter": "3/4", "bars": 8, "fifths": 2, "low": "D4", "high": "D5", "final": "D"}
# fifths: number of sharps (2 = two sharps), or minus the number of flats.

def check_answer(fmt, text, spec):
    if not text.strip():
        print("Paste an answer between the triple quotes first.")
        return None
    text = clean_answer(text)
    result = read_score(fmt, text, scale=35)
    count, unit = (int(x) for x in spec["meter"].split("/"))
    want = {
        "meter": {"count": count, "unit": unit},
        "bars": spec["bars"],
        "fifths": spec["fifths"],
        "low": parse_pitch(spec["low"]),
        "high": parse_pitch(spec["high"]),
        "final": parse_pitch(spec["final"] + "4"),
    }
    if result["ok"]:
        rows = [{"label": "Verovio reads it", "ok": True,
                 "detail": "with warnings, see below the score" if result["log"] else "no warnings"}]
        found = {"meter": result["meter"], "fifths": result["fifths"],
                 "bars": bar_lengths(result["timemap"], result["notes"]), "notes": result["notes"]}
        rows += melody_checks(found, want)
    else:
        rows = [{"label": "Verovio reads it", "ok": False, "detail": " ".join(result["errors"])}]
    width = max(len(r["label"]) for r in rows)
    for r in rows:
        print(f"{'✓' if r['ok'] else '✗'}  {r['label'].ljust(width)}   {r['detail']}")
    print()
    show(result)
    return result

A rendered score under the next cell is the sign that everything works.

In [ ]:
abc = """X:1
T:Alle meine Entchen
M:4/4
L:1/4
K:C
C D E F | G2 G2 | A A A A | G4 |]
"""
result = render("abc", abc)
print("Verovio", result.get("version", "did not run"))

## Part 1: one melody, seven encodings

`render(format, text)` gives the text to Verovio and shows the score with any messages. The format names are Verovio's own (its `inputFrom` option).

### ABC

Letters for pitches, numbers for lengths relative to the unit `L:`. Lower case is the octave above: `c` is one octave above `C`. A sharp is `^F`, a flat `_B`.

In [ ]:
abc = """X:1
T:Alle meine Entchen
M:4/4
L:1/4
K:C
C D E F | G2 G2 | A A A A | G4 |]
"""
result = render("abc", abc)

### Humdrum `**kern`

One line per event, one column ("spine") per voice, separated by tabs. The number before the pitch is the duration: 4 = quarter note, 2 = half note, 1 = whole note. `c` is middle C, `cc` the octave above, `C` the octave below. A sharp is `#`, a flat `-`.

In [ ]:
kern = """!!!OTL: Alle meine Entchen
**kern
*clefG2
*k[]
*M4/4
=1
4c
4d
4e
4f
=2
2g
2g
=3
4a
4a
4a
4a
=4
1g
==
*-
"""
result = render("humdrum", kern)

### Plaine & Easie

Made in 1964 for incipits typed on a typewriter; RISM still uses it. A duration (`4`) stays in force until the next one; `'` sets the octave; `/` is a bar line.

In [ ]:
pae = """@clef:G-2
@keysig:
@timesig:4/4
@data:'4CDEF/2GG/4AAAA/1G//
"""
result = render("pae", pae)

### EsAC

The Essen folksong format: pitches as scale degrees (1 = the key note), `_` lengthens a note. Verovio needs the collection line (`DEMO01`) before `CUT[...]`, and prints two harmless warnings.

In [ ]:
esac = """DEMO01
CUT[Alle meine Entchen]
REG[Europa, Mitteleuropa, Deutschland]
KEY[X0001 04 C 4/4]
MEL[1234 5_5_ 6666 5__ //]
>>
"""
result = render("esac", esac)

### MusicXML

XML for exchange between notation programs. Durations count in `<divisions>` per quarter note.

In [ ]:
musicxml = """<?xml version="1.0" encoding="UTF-8"?>
<score-partwise version="4.0">
  <work><work-title>Alle meine Entchen</work-title></work>
  <part-list><score-part id="P1"><part-name>Voice</part-name></score-part></part-list>
  <part id="P1">
    <measure number="1">
      <attributes><divisions>1</divisions><key><fifths>0</fifths></key>
        <time><beats>4</beats><beat-type>4</beat-type></time>
        <clef><sign>G</sign><line>2</line></clef></attributes>
      <note><pitch><step>C</step><octave>4</octave></pitch><duration>1</duration><type>quarter</type></note>
      <note><pitch><step>D</step><octave>4</octave></pitch><duration>1</duration><type>quarter</type></note>
      <note><pitch><step>E</step><octave>4</octave></pitch><duration>1</duration><type>quarter</type></note>
      <note><pitch><step>F</step><octave>4</octave></pitch><duration>1</duration><type>quarter</type></note>
    </measure>
    <measure number="2">
      <note><pitch><step>G</step><octave>4</octave></pitch><duration>2</duration><type>half</type></note>
      <note><pitch><step>G</step><octave>4</octave></pitch><duration>2</duration><type>half</type></note>
    </measure>
    <measure number="3">
      <note><pitch><step>A</step><octave>4</octave></pitch><duration>1</duration><type>quarter</type></note>
      <note><pitch><step>A</step><octave>4</octave></pitch><duration>1</duration><type>quarter</type></note>
      <note><pitch><step>A</step><octave>4</octave></pitch><duration>1</duration><type>quarter</type></note>
      <note><pitch><step>A</step><octave>4</octave></pitch><duration>1</duration><type>quarter</type></note>
    </measure>
    <measure number="4">
      <note><pitch><step>G</step><octave>4</octave></pitch><duration>4</duration><type>whole</type></note>
      <barline location="right"><bar-style>light-heavy</bar-style></barline>
    </measure>
  </part>
</score-partwise>
"""
result = render("xml", musicxml)

### MEI

The format of digital editions. Verovio converts every format it reads into MEI internally, and `result["mei"]` gives that MEI back. Here we take the MEI made from the kern version, print the part with the notes and render it. Every element has an `xml:id`, so other files can point at a single note.

In [ ]:
mei = read_score("humdrum", kern, svg=False)["mei"]
start = mei.index("<section")
print(mei[start:start + 1500], "…")
result = render("mei", mei)

### GABC (chant, extra)

Gregorian chant: the syllables of the text, each with its neumes in brackets. Letters are positions on the staff, not pitches.

In [ ]:
gabc = """(c4) Al(f)le(g)lu(h)ia.(hg) (::)
"""
result = render("gabc", gabc)

### Sizes and piano rolls

The same four bars take very different amounts of text. And whatever the format, Verovio reads the same notes: the piano roll shows time × pitch, the view from Demo A.

In [ ]:
encodings = {"ABC": abc, "kern": kern, "Plaine & Easie": pae, "EsAC": esac, "MusicXML": musicxml, "MEI": mei}
for name, text in encodings.items():
    print(f"{name:15} {len(text.encode('utf-8')):6,} bytes")

In [ ]:
# Try this: add backend="bokeh" for an interactive roll (drag to move, wheel to zoom, point at a note).
score_roll(read_score("abc", abc, svg=False), title="ABC", height=2.5)
score_roll(read_score("esac", esac, svg=False), title="EsAC", height=2.5)

### Try this

1. Change a note in the ABC and in the kern version above. Add a sharp (`^F` in ABC, `f#` in kern).
2. Change the metre to 3/4 (`M:3/4` in ABC, `*M3/4` in kern). What does Verovio do with bars that no longer fit?
3. Compare what each format records beyond the notes: the phrase brackets in EsAC, the spines in kern, `<divisions>` in MusicXML, the header and the `xml:id` of every element in MEI.
4. Break the syntax on purpose and read the message under the score.

Your own melody (homework 2): encode the first phrase of a melody of your choice here.

In [ ]:
mine = """X:1
T:My melody
M:4/4
L:1/4
K:C
C D E F | G4 |]
"""
result = render("abc", mine)

### When the input is broken

Some broken inputs make Verovio 6.3 crash instead of reporting an error. The helper checks for the known cases first, and it runs Verovio in a separate process, so a crash cannot take the notebook down with it. The first three inputs below would crash Verovio. The last one, Plaine & Easie without its `@data:` key, Verovio rejects itself, and its log says why (lines starting with `Verovio:`).

In [ ]:
broken = [
    ("abc", "T:No X line\nK:C\nCDEF|"),
    ("abc", "X:1\nK:C\nCD]EF|"),
    ("humdrum", "**kern\t**kern\n4c\t4e\n4d\n*-\t*-"),
    ("pae", "@clef:G-2\n'4CDEF/2GG//"),
]
for fmt, text in broken:
    print(f"--- {fmt}: {text!r}")
    render(fmt, text)
    print()

## Part 2: an LLM writes ABC

We asked a language model for a melody and check the answer. Because the answer is an encoding, we can test it, which a picture of a score would not allow.

The prompt, and the follow-up asking for the same melody in kern:

In [ ]:
PROMPT_ABC = ("Write a short melody in ABC notation: 8 bars in 3/4 time, in D major, using only quarter notes, "
              "half notes, dotted half notes and eighth notes. Keep it between D4 and D5 and end on D. "
              "Reply with the ABC only.")
PROMPT_KERN = "Now write the same melody in Humdrum **kern. Reply with the kern only."
print(PROMPT_ABC)
print(PROMPT_KERN)

What the prompt asks for. If you change the prompt, change this too. `fifths` is the key signature: the number of sharps, or minus the number of flats (D major: 2).

In [ ]:
SPEC = {"meter": "3/4", "bars": 8, "fifths": 2, "low": "D4", "high": "D5", "final": "D"}

### Recorded answers

Claude Opus 5.5 (Anthropic) answered both prompts on 2026-10-14, while the course page was built. The answers are not edited.

In [ ]:
claude_abc = """X:1
T:Morning in D
M:3/4
L:1/8
K:D
D2 F2 A2 | d4 c2 | B2 A2 F2 | A6 |
G2 B2 d2 | c2 A2 F2 | E2 F2 E2 | D6 |]
"""
result = check_answer("abc", claude_abc, SPEC)

In [ ]:
claude_kern = """**kern
*clefG2
*k[f#c#]
*M3/4
=1
4d
4f#
4a
=2
2dd
4cc#
=3
4b
4a
4f#
=4
2.a
=5
4g
4b
4dd
=6
4cc#
4a
4f#
=7
4e
4f#
4e
=8
2.d
==
*-
"""
result = check_answer("humdrum", claude_kern, SPEC)

This one was written by hand to show what the checks catch. It is not an LLM answer. Note the Markdown code fence around it: LLMs often answer that way, and `check_answer` removes the fence.

In [ ]:
constructed = """```abc
X:1
T:Waltz in D
M:3/4
L:1/8
K:G
D2 F2 A2 | d3 c B2 A | B2 A2 F2 | A6 |
G2 B2 d2 | e2 c2 A2 | F2 E2 | D6 |]
```
"""
result = check_answer("abc", constructed, SPEC)

### Your turn

Paste an answer between the triple quotes and run the cell. For a kern answer, write `"humdrum"` instead of `"abc"`.

*Does every bar add up to the metre? Does Verovio warn? Are the key and the range what was asked for? Is the kern valid at all?* The checks rely on how Verovio reads the text: Verovio 6.3, for example, ignores a length written after a chord (`[DFA]2`), and the helper warns when it sees one.

In [ ]:
answer = """
"""
result = check_answer("abc", answer, SPEC)